### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [ ]:
# OPTIONAL: Install
!pip install -qU langchain langchain-openai langchain-community langchain-pinecone pinecone python-dotenv tiktoken


## Tutorial: PromptTemplate and Pinecone Retrieval 
We’ll seed a small vector index and retrieve context to feed a clean prompt.

Learning outcomes:
- Design robust `PromptTemplate`s for grounded answers
- Set up Pinecone with OpenAI embeddings
- Retrieve top‑k docs and wire them to the Q&A chain


In [ ]:
import os, re
from typing import Optional, List
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (hidden): ")

# Pinecone setup
INDEX_NAME = os.getenv("PINECONE_INDEX", "lc-demo-index")

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
if INDEX_NAME not in [ix.name for ix in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # matches OpenAI text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=os.getenv("PINECONE_REGION", "us-east-1"))
    )

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", model=MODEL, temperature=0, seed=42)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)


### Step 1: Seed Pinecone with a few texts
We’ll upsert 2‑3 short passages to retrieve from during Q&A.


In [ ]:
texts: List[str] = [
    "LangChain helps developers build LLM applications by composing prompts, chains, tools, and agents.",
    "It emphasizes modularity, integrations, and production‑ready patterns such as retrieval and evaluation.",
    "Prompt templates and retrievers make Q&A more reliable by grounding answers in context."
]
metas = [{"source": "local", "doc_id": f"demo-{i}"} for i in range(len(texts))]
vectorstore.add_texts(texts=texts, metadatas=metas)
print(f"Indexed {len(texts)} texts into Pinecone index '{INDEX_NAME}'.")


### Step 2: Design a robust PromptTemplate for retrieval
We capture source and context explicitly so outputs remain auditable.


In [ ]:
ingest_template = (
    "You answer questions using the provided CONTEXT. Cite the SOURCE if helpful.\n"
    "If the answer is not present, respond: I don't know.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)
ingest_prompt = PromptTemplate.from_template(ingest_template)


### Step 3: Retrieve top‑k from Pinecone and run the chain
We build a context window from retrieved docs and pass it to the prompt.


In [ ]:
ingest_chain = LLMChain(llm=llm, prompt=ingest_prompt)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def build_context(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    return "\n\n".join(d.page_content for d in docs)

q1 = "What is LangChain and one benefit?"
ctx1 = build_context(q1)
print(ingest_chain.run({
    "context": ctx1,
    "question": q1
}))


### Step 4: Try another query
We’ll reuse the retriever to build a new context window.


In [ ]:
q2 = "Name two components LangChain provides to developers."
ctx2 = build_context(q2)
print(ingest_chain.run({
    "context": ctx2,
    "question": q2
}))
